# function_interview_patterns
**Prerequisites:** function_basics, parameters_and_arguments, return_values, scope, closures, lambda_functions, higher_order_functions, map_filter_reduce, recursion

**Outcome:** After this notebook you will recognise the function-related patterns that appear repeatedly in technical interviews, know exactly what the interviewer is testing with each one, and be able to implement clean solutions under pressure.

## Pattern 1 — Memoisation

In [ ]:
# what the interviewer is testing:
# do you know that naive recursion is exponential and how to fix it?

# naive — exponential time O(2^n)
def fib_naive(n):
    if n <= 1:
        return n
    return fib_naive(n - 1) + fib_naive(n - 2)

# memoised — linear time O(n)
def fib_memo(n, cache=None):
    if cache is None:
        cache = {}
    if n in cache:
        return cache[n]
    if n <= 1:
        return n
    cache[n] = fib_memo(n - 1, cache) + fib_memo(n - 2, cache)
    return cache[n]

# cleanest — lru_cache decorator
from functools import lru_cache

@lru_cache(maxsize=None)
def fib(n):
    if n <= 1:
        return n
    return fib(n - 1) + fib(n - 2)

print(fib(50))   # 12586269025 — instant
print(fib.cache_info())

## Pattern 2 — Decorator Implementation

In [ ]:
# what the interviewer is testing:
# do you understand that a decorator is a higher-order function
# that wraps another function and returns the wrapper?

import time
import functools

def timer(func):
    @functools.wraps(func)        # preserves __name__, __doc__ of wrapped function
    def wrapper(*args, **kwargs):
        start  = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} took {elapsed:.6f}s")
        return result
    return wrapper

@timer
def process(records):
    return [{**r, "processed": True} for r in records]

result = process([{"id": f"job_{i}"} for i in range(1000)])
print(f"processed {len(result)} records")
print(process.__name__)   # process — not wrapper, thanks to @functools.wraps

## Pattern 3 — Decorator with Arguments

In [ ]:
# what the interviewer is testing:
# can you write a decorator factory — a function that returns a decorator?

import functools

def retry(max_attempts):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    print(f"attempt {attempt}/{max_attempts} failed: {e}")
                    if attempt == max_attempts:
                        raise
        return wrapper
    return decorator

call_count = 0

@retry(max_attempts=3)
def unstable_fetch(job_id):
    global call_count
    call_count += 1
    if call_count < 3:
        raise ConnectionError("db unreachable")
    return {"id": job_id, "status": "done"}

result = unstable_fetch("job_1")
print(result)

## Pattern 4 — Function Dispatch Table

In [ ]:
# what the interviewer is testing:
# do you know how to eliminate long if/elif chains
# using a dict of functions?

def handle_done(record):
    return {**record, "archived": True}

def handle_failed(record):
    return {**record, "retry": True, "retry_count": record.get("retry_count", 0) + 1}

def handle_pending(record):
    return {**record, "queued": True}

def handle_unknown(record):
    return {**record, "flagged": True}

HANDLERS = {
    "done":    handle_done,
    "failed":  handle_failed,
    "pending": handle_pending,
}

def process(record):
    handler = HANDLERS.get(record["status"], handle_unknown)
    return handler(record)

records = [
    {"id": "job_1", "status": "done"},
    {"id": "job_2", "status": "failed"},
    {"id": "job_3", "status": "pending"},
    {"id": "job_4", "status": "unknown_state"},
]

for r in records:
    print(process(r))

## Pattern 5 — Partial Application

In [ ]:
# what the interviewer is testing:
# do you know how to pre-fill arguments to create specialised functions?

from functools import partial

def send_alert(level, region, message):
    print(f"[{level}] {region}: {message}")

# create specialised versions without rewriting the function
critical_us = partial(send_alert, "CRITICAL", "us-east")
warning_eu  = partial(send_alert, "WARNING",  "eu-west")

critical_us("db connection lost")
warning_eu("latency above threshold")
critical_us("disk full")

# equivalent closure approach — same result, more explicit
def make_alerter(level, region):
    def alert(message):
        send_alert(level, region, message)
    return alert

info_ap = make_alerter("INFO", "ap-south")
info_ap("pipeline started")

## Pattern 6 — Accumulator with Closure

In [ ]:
# what the interviewer is testing:
# can you build stateful behaviour without a class
# using closures and nonlocal?

def make_running_average():
    total = 0
    count = 0

    def update(value):
        nonlocal total, count
        total += value
        count += 1
        return total / count

    return update

avg = make_running_average()

print(avg(100))   # 100.0
print(avg(200))   # 150.0
print(avg(300))   # 200.0
print(avg(80))    # 170.0

## Pattern 7 — Single Responsibility Refactor

In [ ]:
# what the interviewer is testing:
# can you look at a bloated function and break it into
# single-responsibility pieces that are testable in isolation?

# before — one function doing everything
def run_pipeline(raw_data, output_path, region, dry_run=False):
    valid = [r for r in raw_data if r.get("id") and r.get("value") is not None]
    enriched = [{**r, "region": region, "status": "processed"} for r in valid]
    sorted_data = sorted(enriched, key=lambda r: r["id"])
    if not dry_run:
        with open(output_path, "w") as f:
            for r in sorted_data:
                f.write(str(r) + "\n")
    return sorted_data

# after — each function has one job, each is independently testable
def validate(records):
    return [r for r in records if r.get("id") and r.get("value") is not None]

def enrich(records, region):
    return [{**r, "region": region, "status": "processed"} for r in records]

def sort_by_id(records):
    return sorted(records, key=lambda r: r["id"])

def write_output(records, output_path):
    with open(output_path, "w") as f:
        for r in records:
            f.write(str(r) + "\n")

def run_pipeline(raw_data, output_path, region, dry_run=False):
    result = sort_by_id(enrich(validate(raw_data), region))
    if not dry_run:
        write_output(result, output_path)
    return result

raw = [
    {"id": "job_3", "value": 30},
    {"id": "job_1", "value": None},
    {"id": "job_2", "value": 20},
]

result = run_pipeline(raw, "/tmp/out.txt", "us-east", dry_run=True)
for r in result:
    print(r)

## Pattern 8 — Recursive Tree Traversal

In [ ]:
# what the interviewer is testing:
# can you traverse a tree structure of unknown depth
# and aggregate values from all nodes?

org = {
    "name": "engineering",
    "headcount": 5,
    "teams": [
        {
            "name": "platform",
            "headcount": 8,
            "teams": [
                {"name": "infra",    "headcount": 4, "teams": []},
                {"name": "security", "headcount": 3, "teams": []},
            ]
        },
        {
            "name": "product",
            "headcount": 6,
            "teams": [
                {"name": "frontend", "headcount": 5, "teams": []},
            ]
        }
    ]
}

def total_headcount(node):
    count = node["headcount"]
    for team in node["teams"]:
        count += total_headcount(team)
    return count

def all_team_names(node):
    names = [node["name"]]
    for team in node["teams"]:
        names.extend(all_team_names(team))
    return names

print(f"total headcount : {total_headcount(org)}")   # 31
print(f"all teams       : {all_team_names(org)}")

## Pattern 9 — Generator as Lazy Pipeline

In [ ]:
# what the interviewer is testing:
# do you know how to process large sequences
# one element at a time without loading everything into memory?

def read_records(n):
    for i in range(n):
        yield {"id": f"job_{i}", "value": i, "status": "raw"}

def filter_positive(records):
    for r in records:
        if r["value"] > 0:
            yield r

def enrich(records):
    for r in records:
        yield {**r, "status": "enriched"}

# pipeline — nothing computed until consumed
pipeline = enrich(filter_positive(read_records(1_000_000)))

# consume only the first 5
for record in pipeline:
    print(record)
    if record["value"] >= 5:
        break

# memory used: O(1) — only one record in memory at a time

## Pattern 10 — Pure vs Impure Functions

In [ ]:
# what the interviewer is testing:
# do you understand the difference between pure functions
# (same input always gives same output, no side effects)
# and impure functions (modify state, I/O, non-deterministic)?

# impure — reads external state, result depends on call order
processed_ids = set()

def process_impure(record):
    if record["id"] in processed_ids:   # external state
        return None
    processed_ids.add(record["id"])      # side effect
    return {**record, "done": True}

# pure — same input always gives same output, no side effects
def process_pure(record, seen_ids):
    if record["id"] in seen_ids:
        return None, seen_ids
    new_seen = seen_ids | {record["id"]}   # return new state, do not mutate
    return {**record, "done": True}, new_seen

seen = set()
records = [
    {"id": "job_1"},
    {"id": "job_1"},   # duplicate
    {"id": "job_2"},
]

for record in records:
    result, seen = process_pure(record, seen)
    print(result)

## Common Interview Questions on Functions

- What is the difference between `*args` and `**kwargs`?
- What is a closure and what problem does it solve?
- What is a decorator and how do you implement one?
- What is `functools.wraps` and why do you need it?
- What is memoisation and when would you use `lru_cache`?
- What is the difference between a pure and an impure function?
- Why does Python not support tail call optimisation?
- What is partial application and how does `functools.partial` work?
- What is a dispatch table and when is it better than if/elif?

## Exercise

In [ ]:
import functools

# implement a decorator called 'validate_input' that:
# - takes a predicate function as its argument
# - wraps any single-argument function
# - raises ValueError with message 'invalid input: <value>'
#   if predicate(argument) returns False
# - calls and returns the wrapped function normally if predicate passes
# - preserves the wrapped function's __name__

def validate_input(predicate):
    pass   # implement here


# usage
is_positive    = lambda x: x > 0
is_non_empty   = lambda x: bool(x)

@validate_input(is_positive)
def compute_retry_delay(attempt):
    return 2 ** attempt

@validate_input(is_non_empty)
def process_batch(records):
    return [{**r, "done": True} for r in records]


assert compute_retry_delay(3)  == 8
assert compute_retry_delay(1)  == 2
assert compute_retry_delay.__name__ == "compute_retry_delay"

assert process_batch([{"id": "job_1"}]) == [{"id": "job_1", "done": True}]

try:
    compute_retry_delay(-1)
    assert False, "should have raised"
except ValueError as e:
    assert "invalid input" in str(e), f"got: {e}"
    assert "-1" in str(e),            f"got: {e}"

try:
    process_batch([])
    assert False, "should have raised"
except ValueError as e:
    assert "invalid input" in str(e), f"got: {e}"

print("all assertions passed")